# Two ways to grow a vein network

![Two vein recipes on the same leaf](../assets/vein_models_preview.png)

A leaf has to deliver water to every cell. Two very different recipes can produce a network that does it, and this notebook lets you grow both on the same leaf and measure them with the same ruler. Sources are listed at the end.

**Recipe A: grow toward the cells** (space colonization, Runions et al. 2005 [1]). Scatter "auxin sources" through the leaf. Veins creep toward them and a source disappears when a vein arrives. There is no water in this recipe, only geometry.

**Recipe B: let the flow decide** (adaptive flow network, Hu and Cai 2013 [2]; moving load after Katifori et al. 2010 [3], Corson 2010 [4]). Start with every possible pipe in place, all equally thick. Pump water in at the base and let it drain everywhere. Busy pipes thicken, idle ones wither. There is no geometry in this recipe beyond the mesh it sits on, only flow and cost.

| | Recipe A (space colonization) | Recipe B (adaptive flow) |
|:--|:--|:--|
| Starts from | a leaf outline, scattered sources | a mesh of pipes inside the outline |
| Veins appear by | growing toward sources | pipes surviving |
| Tree or loops decided by | a rule: *open* or *closed* | physics: cost exponent γ and how the load moves |
| Widths come from | a rule afterwards (Murray's law) | the flow itself |
| Main knob | source spacing $b$, kill distance $d_k$ | upkeep exponent $\gamma$, load fluctuation $\alpha$ |

**How to use:** run top to bottom (needs `ipywidgets`). Sliders set up a run, **Run** grows it, and the *time* slider scrubs through the growth afterwards. Everything is on the same dark canvas and the same colour scale, thin veins dim and thick veins bright.

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display

from math_vis import finish
from pattern_formation.networks import relax_moving_graph, moving_load_Q2, count_loops
from pattern_formation.venation import (
    SHAPES, PALETTE, BG, leaf_outline, grow_veins, leaf_mesh, murray_widths,
    vein_stats, draw_veins, render_veins,
)

# Tests run this notebook as plain Python; keep everything tiny there.
FAST = "PYTEST_CURRENT_TEST" in os.environ

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": "#d9e2dc", "axes.labelcolor": "#d9e2dc", "axes.edgecolor": "#3a4a40",
    "xtick.color": "#9fb0a5", "ytick.color": "#9fb0a5",
})


def slider(kind, **kw):
    return kind(style={"description_width": "10em"}, layout=w.Layout(width="28em"), **kw)


def leaf_dropdown():
    return w.Dropdown(options=SHAPES, value="ovate", description="leaf shape",
                      style={"description_width": "10em"}, layout=w.Layout(width="28em"))

In [3]:
print(1)

1


## 1. Recipe A: veins grow toward auxin sources

**Sources.** Dart throwing: random points inside the leaf, never closer than the **birth distance** $b$ to another source or to a vein. $b$ sets the spacing of the finest veins.

**Growth.** Every source belongs to its nearest vein node. A node with sources takes one step of length $D$ toward the *average direction* of its sources:

$$\mathbf n = \frac{\sum_s \hat{\mathbf u}_s}{\left\lVert \sum_s \hat{\mathbf u}_s \right\rVert}, \qquad \mathbf x_\text{new} = \mathbf x + D\,\mathbf n, \qquad \hat{\mathbf u}_s = \frac{\mathbf x_s - \mathbf x}{\lVert \mathbf x_s - \mathbf x \rVert}.$$

**Kill.** A source is removed once a vein comes within the **kill distance** $d_k$.

**Open or closed.** *Open*: each source belongs to its nearest vein node, so the result is always a tree. *Closed*: a source belongs to every node that is its **relative neighbour** (Runions et al. [1], eq. 2: no other node is closer to both), so several veins can head for the same source. In the paper the source stays alive until every vein growing toward it has arrived or left its zone of influence. Here the relative neighbours are recomputed every step rather than passed on to descendants as tags, and when all have arrived they are joined by a new node at the source, which makes loops. *The text of the paper I read does not say how veins are joined at the source; that rule is mine.*

**Width.** Murray's law [5], from the tips back to the base: $r_\text{parent}^{\,n} = \sum r_\text{child}^{\,n}$, tips have $r = 1$. With $n = 2$ the root radius is $\sqrt{\#\text{tips}}$ exactly.

**Leaf growth.** The leaf starts small and its outline scales up about the base while sources keep appearing in the new margin. Lengths are in units of the final leaf length, so the step is $D = 0.01$.

Left: the leaf at the chosen time, with line width and colour set by Murray radius. Right: loops over time.

In [4]:
def run_A(shape, b, kill, closed, growth, seed, steps):
    poly = leaf_outline(shape)
    g = grow_veins(poly, b=b, kill=kill, closed=closed, start_size=0.3 if growth else 1.0,
                   steps=steps, grow_steps=steps * 2 // 5, seed=seed)
    return g


def loops_over_time(g, n_exp):
    ts = list(range(0, len(g["n_nodes"]), max(1, len(g["n_nodes"]) // 50)))
    out = []
    for t in ts:
        nn, ne = g["n_nodes"][t], g["n_edges"][t]
        _, wd = murray_widths(g["parent"][:nn], g["edges"][:ne], n=n_exp)
        out.append(vein_stats(g["xy"][:nn], g["edges"][:ne], wd, 0.5)["loops"])
    return ts, out


def show_A(g, t, n_exp, show_sources):
    nn, ne = g["n_nodes"][t], g["n_edges"][t]
    xy, edges = g["xy"][:nn], g["edges"][:ne]
    r, wd = murray_widths(g["parent"][:nn], edges, n=n_exp)
    st = vein_stats(xy, edges, wd, 0.5, root=0)
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 5.2), gridspec_kw={"width_ratios": [1, 1.15]})
    poly = g["poly"] * g["size"][t]
    draw_veins(a, xy, edges, wd, poly, lw=5.0)
    if show_sources and len(g["sources"][t]):
        a.scatter(*g["sources"][t].T, s=6, color="#ff9d5c", zorder=4)
    a.set_title(f"step {t}", color="#d9e2dc")
    ts, loops = loops_over_time(g, n_exp)
    b.plot(ts, loops, color="#d8d26a")
    b.axvline(t, color="#ff9d5c")
    b.set_xlabel("growth step")
    b.set_ylabel("independent loops")
    plt.tight_layout()
    plt.show()
    leaves = st["tips"]
    print(f"nodes {nn}   links {st['links']}   tips {leaves}   loops {st['loops']}   "
          f"total vein length {st['length']:.2f} (leaf length 1)")
    if st["loops"] == 0 and ne:
        print(f"Murray: root radius {r[0]:.3f}   tips^(1/n) = {leaves ** (1 / n_exp):.3f}")


ctrl_A = dict(
    shape=leaf_dropdown(),
    closed=w.ToggleButtons(options=[("open (tree)", False), ("closed (loops)", True)],
                           value=False, description="venation"),
    b=slider(w.FloatSlider, value=0.05, min=0.025, max=0.12, step=0.005,
             readout_format=".3f", description="birth distance b"),
    kill=slider(w.FloatSlider, value=0.02, min=0.01, max=0.08, step=0.005,
                readout_format=".3f", description="kill distance d_k"),
    growth=w.Checkbox(value=True, description="leaf grows while veins form",
                      layout=w.Layout(width="28em")),
    steps=slider(w.IntSlider, value=60 if FAST else 200, min=40, max=300, step=20,
                 description="steps"),
    seed=slider(w.IntSlider, value=0, min=0, max=9, description="seed"),
    n_exp=slider(w.FloatSlider, value=2.0, min=1.5, max=3.5, step=0.5, description="Murray exponent n"),
    src=w.Checkbox(value=False, description="show auxin sources", layout=w.Layout(width="28em")),
)
runA = w.Button(description="Grow", button_style="primary")
statusA = w.HTML()
frameA = slider(w.IntSlider, value=0, min=0, max=199, description="time", continuous_update=False)
outA = w.Output()
SA = {}


def redrawA(*_):
    g = SA.get("g")
    if g is None:
        return
    with outA:
        outA.clear_output(wait=True)
        show_A(g, min(frameA.value, len(g["n_nodes"]) - 1), ctrl_A["n_exp"].value, ctrl_A["src"].value)


def on_runA(_):
    runA.disabled = True
    statusA.value = "growing..."
    t0 = time.time()
    c = ctrl_A
    SA["g"] = run_A(c["shape"].value, c["b"].value, c["kill"].value, c["closed"].value,
                    c["growth"].value, c["seed"].value, c["steps"].value)
    last = len(SA["g"]["n_nodes"]) - 1
    frameA.max = last
    statusA.value = f"done in {time.time() - t0:.1f} s"
    runA.disabled = False
    if frameA.value == last:
        redrawA()
    else:
        frameA.value = last


runA.on_click(on_runA)
frameA.observe(redrawA, "value")
ctrl_A["n_exp"].observe(redrawA, "value")
ctrl_A["src"].observe(redrawA, "value")
display(w.VBox(list(ctrl_A.values()) + [w.HBox([runA, statusA]), frameA, outA]))
on_runA(None)


**Question 1.** Open venation, $b = 0.05$. Raise the kill distance from 0.02 to 0.08. What happens to the number of tips?

<details><summary>Answer</summary>

On the ovate leaf (3 seeds) the tip count falls from about 60 to about 25. Veins stop growing as soon as they are within $d_k$ of a source, so a large $d_k$ stops the fine branches from forming.

</details>

**Question 2.** Switch to closed and raise $d_k$ from 0.01 to 0.08. Do loops go up or down?

<details><summary>Answer</summary>

Up in this implementation: about 30 loops at 0.01 and about 115 at 0.08 (4 seeds, $b = 0.05$). **This is not the paper's result.** Fig. 9a-e of Runions et al. [1] shows the opposite: a smaller kill distance gives a denser net. Once $d_k$ approaches $b$ a source counts as reached almost as soon as it is born, and my joining rule then links nearby veins to it, which adds length and loops. The trend comes from my joining rule, not from the mechanism in the paper.

</details>

## 2. Recipe B: pipes adapt to the flow

The leaf is covered with a mesh of nodes joined by pipes of conductance $C$ and length $\ell$. The mesh is a jittered triangular lattice joined by a Delaunay graph, so no direction is special and a node has about 6 pipes (a square grid forces veins along two directions, as an earlier version of this notebook showed). Fluid enters at the base and every other node drains.

**Step 1: flow follows pressure.** Flow in equals flow out at every node:

$$\sum_j \frac{C_{ij}}{\ell_{ij}}\,(p_i - p_j) = s_i, \qquad Q_{ij} = \frac{C_{ij}}{\ell_{ij}}\,(p_i - p_j).$$

**Step 2: pipes adapt.**

$$\frac{dC}{dt} = \underbrace{\frac{Q^2}{C}}_{\text{busy pipes widen}} - \underbrace{\nu\,C^{\gamma}}_{\text{upkeep}}, \qquad \nu = 1.$$

This is gradient descent, with a positive factor per pipe, on the energy $E = \tfrac12\sum \ell\,(Q^2/C + \nu C^\gamma/\gamma)$: pumping power plus upkeep [2, 6]. Any positive factor settles in the same state, so I chose the simplest; Hu and Cai's own update may use a different factor. The step is implicit, because the plain explicit step is unstable for thin pipes and an earlier version of this notebook made them flicker on and off.

**What a settled pipe looks like.** Growth equals upkeep, so $C = (Q^2/\nu)^{1/(\gamma+1)}$, that is $\log C$ against $\log Q$ is a straight line of slope $2/(\gamma+1)$. For a round pipe $C \propto r^4$, so with $\gamma = \tfrac12$ the radius obeys $r^3 \propto Q$. Flow adds up where pipes meet, so **$r^3$ adds up too: Murray's law with $n = 3$**, here from flow instead of imposed.

**Tree or loops.** With a steady load, $\gamma < 1$ gives a tree and $\gamma > 1$ keeps loops [3, 4]. A *moving* load keeps loops even for $\gamma < 1$ [3, 4]. Here the load follows Klemm and Martens [6]: the base supplies 1 unit, every sink drains $(1-\alpha)/m$, and one of the $m$ sinks, chosen at random, drains $\alpha$ more. I average over all $m$ choices exactly instead of drawing one, so a run has no noise. $\alpha = 0$ is the steady load, $\alpha = 1$ is one sink open at a time. Link lengths do not move the settled state (the pipe's upkeep and power both scale with $\ell$), but they let the drawing be true to the mesh.

Left: the network at the chosen time (colour and thickness by radius $r = C^{1/4}$, links below the cutoff $10^x C_{\max}$ hidden; the finest settled pipes sit near $10^{-3}C_{\max}$, so keep $x$ near $-4$). Right: loops over time. The readout compares the measured slope with $2/(\gamma+1)$ once the network has settled (about 800 steps).

In [ ]:
DT = 0.5  # implicit step, so large steps are stable


def run_B(shape, rows, gamma, alpha, steps, seed, frames=30):
    poly = leaf_outline(shape)
    xy, edges, base = leaf_mesh(poly, rows, seed=seed)
    N = len(xy)
    length = np.linalg.norm(xy[edges[:, 0]] - xy[edges[:, 1]], axis=1)
    sinks = np.array([i for i in range(N) if i != base])
    chunk = max(1, steps // frames)
    C = np.ones(len(edges))
    times, snaps = [0], [C]
    for _ in range(steps // chunk):
        C = relax_moving_graph(edges, N, base, sinks, alpha, gamma=gamma, dt=DT, steps=chunk,
                               C0=C, length=length)
        times.append(times[-1] + chunk)
        snaps.append(C)
    return dict(poly=poly, xy=xy, edges=edges, length=length, sinks=sinks, times=times,
                snaps=snaps, gamma=gamma, alpha=alpha, base=base)


def slope_vs_prediction(run, C):
    # at a settled pipe C = (<Q^2>/nu)^(1/(gamma+1)), with <Q^2> the load average
    Q = np.sqrt(moving_load_Q2(run["edges"], C, len(run["xy"]), run["base"], run["sinks"],
                               run["alpha"], run["length"]))
    k = C > 1e-2 * C.max()
    if k.sum() < 5 or np.ptp(np.log(Q[k])) < 1e-6:
        return None
    return np.polyfit(np.log(Q[k]), np.log(C[k]), 1)[0]


def show_B(run, i, cut_exp):
    xy, edges, poly = run["xy"], run["edges"], run["poly"]
    N = len(xy)
    C = run["snaps"][i]
    cut = 10.0 ** cut_exp * C.max()
    loops = [vein_stats(xy, edges, c, 10.0 ** cut_exp * c.max())["loops"] for c in run["snaps"]]
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 5.2), gridspec_kw={"width_ratios": [1, 1.15]})
    draw_veins(a, xy, edges, C ** 0.25, poly, cutoff=cut ** 0.25, lw=5.0, power=3.0)
    a.set_title(f"after {run['times'][i]} adaptation steps", color="#d9e2dc")
    b.plot(run["times"], loops, color="#d8d26a")
    b.axhline(len(edges) - N + 1, ls="--", color="#4a5d51")
    b.axvline(run["times"][i], color="#ff9d5c")
    b.set_xlabel("adaptation steps")
    b.set_ylabel("independent loops (dashed: most possible)")
    plt.tight_layout()
    plt.show()
    st = vein_stats(xy, edges, C, cut, root=run["base"])
    print(f"links kept {st['links']} of {len(edges)}   tips {st['tips']}   loops {st['loops']}   "
          f"(a tree on {N} nodes has {N - 1} links)")
    sl = slope_vs_prediction(run, C)
    if sl is not None:
        print(f"slope of log C vs log Q: {sl:.3f}   predicted 2/(gamma+1) = {2 / (run['gamma'] + 1):.3f}")


ctrl_B = dict(
    shape=leaf_dropdown(),
    gamma=slider(w.FloatSlider, value=0.5, min=0.3, max=1.5, step=0.05, description="gamma (upkeep exponent)"),
    alpha=slider(w.FloatSlider, value=0.6, min=0.0, max=1.0, step=0.1,
                 description="alpha (0 steady, 1 one sink)"),
    rows=slider(w.IntSlider, value=12 if FAST else 28, min=12, max=40, description="mesh rows"),
    steps=slider(w.IntSlider, value=100 if FAST else 800, min=100, max=3000, step=100, description="steps"),
    seed=slider(w.IntSlider, value=0, min=0, max=9, description="mesh seed"),
)
runB = w.Button(description="Run", button_style="primary")
statusB = w.HTML()
frameB = slider(w.IntSlider, value=0, min=0, max=30, description="time", continuous_update=False)
cutB = slider(w.FloatSlider, value=-4, min=-6, max=-1, step=0.25, description="cutoff exponent x",
              continuous_update=False)
outB = w.Output()
SB = {}


def redrawB(*_):
    run = SB.get("run")
    if run is None:
        return
    with outB:
        outB.clear_output(wait=True)
        show_B(run, min(frameB.value, len(run["times"]) - 1), cutB.value)


def on_runB(_):
    runB.disabled = True
    statusB.value = "running..."
    t0 = time.time()
    c = ctrl_B
    SB["run"] = run_B(c["shape"].value, c["rows"].value, c["gamma"].value, c["alpha"].value,
                      c["steps"].value, c["seed"].value)
    last = len(SB["run"]["times"]) - 1
    frameB.max = last
    statusB.value = f"done in {time.time() - t0:.1f} s"
    runB.disabled = False
    if frameB.value == last:
        redrawB()
    else:
        frameB.value = last


runB.on_click(on_runB)
frameB.observe(redrawB, "value")
cutB.observe(redrawB, "value")
display(w.VBox(list(ctrl_B.values()) + [w.HBox([runB, statusB]), frameB, cutB, outB]))
on_runB(None)

**Question 3.** Steady load ($\alpha = 0$) on the ovate leaf, 800 steps. Try $\gamma = 0.9$ and then $\gamma = 1.1$. What changes?

<details><summary>Answer</summary>

At $\gamma = 0.9$ the settled network is a spanning tree: 0 loops and $N - 1$ links (about 270). At $\gamma = 1.1$ it keeps about 165 loops (162, 170 and 166 on three meshes) out of about 500 possible. The switch sits at $\gamma = 1$, as the cost argument above predicts [3, 4].

</details>

**Question 4.** $\gamma = 0.5$. Raise $\alpha$ from 0 to 1. How many loops?

<details><summary>Answer</summary>

Loops appear even though the steady load at this $\gamma$ gives a tree: 0 at $\alpha = 0$, about 50 at 0.3, about 65 at 0.6 and about 74 at 1.0 (three meshes each, cutoff $10^{-4}C_{\max}$). Tips fall from about 60 to about 10. Only the fluctuation made the loops here [3, 4].

</details>

**Question 5.** Change the mesh seed from 0 to 5 at $\alpha = 0.6$. How much does the picture change, and how much does the loop count?

<details><summary>Answer</summary>

The loop count hardly moves (62 to 70 over seeds 0 to 5), but every mesh gives a different net: the count is a property of the recipe, the layout of the veins is not.

</details>

## 3. Same leaf, same ruler

Four networks on the same leaf (the two flow networks share one mesh). All are measured by the same function: independent loops $= E - N + K$ (links, nodes, connected pieces), tips (nodes with one link), and total length. The last panel compares how widths are distributed, each network normalised by its own widest vein.

In [ ]:
def compare(shape, b, gamma, rows, steps_flow):
    poly = leaf_outline(shape)
    items = []
    for name, closed, kill in [("A · open", False, 0.02), ("A · closed", True, 0.01)]:
        g = grow_veins(poly, b=b, kill=kill, closed=closed, steps=60 if FAST else 200)
        r, wd = murray_widths(g["parent"], g["edges"])
        items.append((name, g["xy"], g["edges"], wd, 0.5, dict(power=0.8)))
    for name, alpha in [("B · steady flow", 0.0), ("B · moving flow", 0.6)]:
        run = run_B(shape, rows, gamma, alpha, steps_flow, 0, frames=1)
        C = run["snaps"][-1]
        items.append((name, run["xy"], run["edges"], C ** 0.25, (1e-4 * C.max()) ** 0.25,
                      dict(power=3.0)))
    base = run["base"]
    fig = plt.figure(figsize=(14, 7.2))
    gs = fig.add_gridspec(2, 4, height_ratios=[4, 1.5])
    print(f"{'network':18s}{'links':>8s}{'tips':>8s}{'loops':>8s}{'length':>9s}")
    for j, (name, xy, edges, wd, cut, kw) in enumerate(items):
        ax = fig.add_subplot(gs[0, j])
        draw_veins(ax, xy, edges, wd, poly, cutoff=cut, lw=5.0, **kw)
        ax.set_title(name, color="#d9e2dc")
        st = vein_stats(xy, edges, wd, cut, root=0 if name.startswith("A") else base)
        print(f"{name:18s}{st['links']:8d}{st['tips']:8d}{st['loops']:8d}{st['length']:9.2f}")
    ax = fig.add_subplot(gs[1, :])
    for (name, xy, edges, wd, cut, kw), col in zip(items, ["#6fae3f", "#d8d26a", "#20a4c9", "#f08a5d"]):
        x = wd[wd > cut] / wd.max()
        ax.hist(x, bins=np.linspace(0, 1, 41), histtype="step", color=col, lw=1.6, label=name)
    ax.set_yscale("log")
    ax.set_xlabel("vein radius / widest vein")
    ax.set_ylabel("links")
    ax.legend(frameon=False, ncol=4)
    plt.tight_layout()
    plt.show()


cmp_shape = leaf_dropdown()
cmp_b = slider(w.FloatSlider, value=0.05, min=0.03, max=0.1, step=0.005, readout_format=".3f",
               description="b (recipe A)")
cmp_g = slider(w.FloatSlider, value=0.5, min=0.3, max=1.6, step=0.1, description="gamma (recipe B)")
cmp_btn = w.Button(description="Grow all four (~10 s)", button_style="primary")
cmp_out = w.Output()


def on_cmp(_):
    cmp_btn.disabled = True
    with cmp_out:
        cmp_out.clear_output(wait=True)
        print("growing...")
        compare(cmp_shape.value, cmp_b.value, cmp_g.value, 12 if FAST else 28, 100 if FAST else 800)
    cmp_btn.disabled = False


cmp_btn.on_click(on_cmp)
display(w.VBox([cmp_shape, cmp_b, cmp_g, cmp_btn, cmp_out]))
if FAST:
    on_cmp(None)

## 4. Numbers the calculations predicted

Five checks, each a number you can compute by hand.

1. **Murray's law (Recipe A).** The root radius must equal $(\#\text{tips})^{1/n}$ exactly. This holds for every leaf shape and $n$.
2. **The flow power law (Recipe B).** For settled pipes, $\log C$ against $\log Q$ has slope $2/(\gamma+1)$. At $\gamma = \tfrac12$ that is $r^3 \propto Q$: Murray's law with $n=3$. It holds for a moving load too, with $Q$ the root-mean-square flow over the load.
3. **A tree has $N - 1$ links (Recipe B).** A steady load with $\gamma < 1$ must settle into a spanning tree: $N-1$ links, no loops. With $\gamma > 1$ it must not.
4. **A three-node loop (Recipe B).** One source, two sinks, $\gamma = 1$ (Klemm and Martens [6]). The loop link must stay shut for $\alpha < 1/\sqrt3 \approx 0.577$ and open above it, with $C = (\alpha - 1/\sqrt3)/2$.
5. **Spacing (Recipe A).** The only length that sets the mesh is $b$, so the number of loops in closed venation should scale as $1/b^2$. This one is *rough*: see below.

In [ ]:
def check_numbers():
    print("1. Murray's law, open venation: root radius vs tips^(1/n)")
    print(f"   {'shape':12s}{'tips':>6s}  n=2 measured / predicted   n=3 measured / predicted")
    for shape in SHAPES:
        g = grow_veins(leaf_outline(shape), b=0.05, kill=0.02, closed=False, steps=60 if FAST else 200)
        par = g["parent"]
        tips = int((np.bincount(par[par >= 0], minlength=len(par)) == 0).sum())
        row = ""
        for n in (2.0, 3.0):
            r, _ = murray_widths(par, g["edges"], n=n)
            row += f"   {r[0]:9.4f} / {tips ** (1 / n):9.4f}       "
        print(f"   {shape:12s}{tips:6d}{row}")

    print("\n2. Flow power law on a leaf mesh: slope of log C against log Q")
    xy, edges, base = leaf_mesh(leaf_outline("ovate"), 12 if FAST else 24)
    N = len(xy)
    length = np.linalg.norm(xy[edges[:, 0]] - xy[edges[:, 1]], axis=1)
    sinks = np.array([i for i in range(N) if i != base])
    steps = 100 if FAST else 800
    for alpha, gamma in [(0.0, 0.5), (0.0, 0.75), (0.6, 0.5)]:
        C = relax_moving_graph(edges, N, base, sinks, alpha, gamma=gamma, dt=DT, steps=steps, length=length)
        Q = np.sqrt(moving_load_Q2(edges, C, N, base, sinks, alpha, length))
        k = C > 1e-2 * C.max()
        slope = np.polyfit(np.log(Q[k]), np.log(C[k]), 1)[0]
        print(f"   alpha {alpha}, gamma {gamma}: slope {slope:.3f}   predicted {2 / (gamma + 1):.3f}")

    print(f"\n3. Steady load: links kept (a spanning tree has N - 1 = {N - 1})")
    for gamma in (0.5, 1.2):
        C = relax_moving_graph(edges, N, base, sinks, 0.0, gamma=gamma, dt=DT, steps=steps, length=length)
        st = vein_stats(xy, edges, C, 1e-4 * C.max(), root=base)
        print(f"   gamma {gamma}: links {st['links']}   loops {st['loops']}")

    print("\n4. Three-node loop at gamma 1: third link C against (alpha - 1/sqrt 3) / 2")
    tri = np.array([[0, 1], [0, 2], [1, 2]])
    for alpha in (0.4, 0.7, 1.0):
        C = relax_moving_graph(tri, 3, 0, np.array([1, 2]), alpha, gamma=1.0, dt=0.2, steps=6000)
        print(f"   alpha {alpha}: C = {C[2]:.4f}   predicted {max((alpha - 3 ** -0.5) / 2, 0):.4f}")

    print("\n5. Closed venation, loops vs b (no leaf growth, mean of 3 seeds)")
    bs = [0.06, 0.1] if FAST else [0.04, 0.055, 0.075, 0.1, 0.14]
    L = []
    for b in bs:
        ls = []
        for sd in range(3):
            g = grow_veins(leaf_outline("ovate"), b=b, kill=0.01, closed=True, start_size=1.0,
                           steps=60 if FAST else 250, seed=sd)
            _, wd = murray_widths(g["parent"], g["edges"])
            ls.append(vein_stats(g["xy"], g["edges"], wd, 0.5)["loops"])
        L.append(np.mean(ls))
        print(f"   b {b:5.3f}: loops {L[-1]:6.1f}   loops x b^2 = {L[-1] * b * b:.3f}")
    if min(L) > 0:
        print(f"   fitted exponent {np.polyfit(np.log(bs), np.log(L), 1)[0]:.2f}   predicted -2")


num_btn = w.Button(description="Run the checks (~15 s)", button_style="primary")
num_out = w.Output()


def on_num(_):
    num_btn.disabled = True
    with num_out:
        num_out.clear_output(wait=True)
        print("running...")
        check_numbers()
    num_btn.disabled = False


num_btn.on_click(on_num)
display(w.VBox([num_btn, num_out]))
if FAST:
    on_num(None)

**What held and what did not.** Checks 1 to 4 hold (the flow slope is 1.333 at $\gamma=\tfrac12$ and 1.143 at 0.75 to the printed digits, the steady tree has exactly $N-1$ links, and the three-node loop opens at $1/\sqrt3$ with $C = (\alpha-1/\sqrt3)/2$). Checks 1, 2 and 4 are tests in the repository. Check 5 does not give $-2$: over $b = 0.04$ to $0.14$ the fitted exponent is between $-3.0$ (5 seeds) and $-3.5$ (the 3 seeds in the cell above). At large $b$ only a handful of loops are left in a leaf a few $b$ wide, so finite size is a likely reason, but I have not shown that. Is the variation between seeds the whole story, or does something else set the loop count?

**Two bugs found while checking against the papers.** (1) In space colonization, a vein node pulled by two sources in nearly opposite directions steps sideways and gets no closer to either, so it stacked copies of the same child on one spot every step; about 40 % of nodes were duplicates, which inflated tip counts (440 became 60) and faked loops in closed venation. (2) The explicit adaptation step was unstable for thin pipes, so they flickered instead of dying and the steady-load loop count never settled. Both are fixed and tested.

## 5. Gallery

The same closed venation on every leaf shape, drawn with the `math_vis` finishing pass (grain) and a palette of your choice. Try other `b` values for a finer or coarser net.

In [ ]:
PALETTES = {
    "leaf": PALETTE,
    "dusk": ["#1b0b2e", "#5a1a6b", "#c2417a", "#f08a5d", "#ffd6a5"],
    "ice": ["#06141f", "#0b4f6c", "#20a4c9", "#9be3e8", "#f2fdff"],
    "ember": ["#2b0a0a", "#7a1f12", "#d9531e", "#f4b942", "#fff2c4"],
}


def gallery(b, palette, closed):
    fig, axs = plt.subplots(1, len(SHAPES), figsize=(15, 4.6))
    for ax, shape in zip(axs, SHAPES):
        poly = leaf_outline(shape)
        g = grow_veins(poly, b=b, kill=0.01 if closed else 0.02, closed=closed,
                       steps=60 if FAST else 200)
        _, wd = murray_widths(g["parent"], g["edges"])
        img = render_veins(g["xy"], g["edges"], wd, poly, dpi=40 if FAST else 130,
                           palette=PALETTES[palette], lw=5.0)
        ax.imshow(finish(img, grain=2.0, display=False))
        ax.axis("off")
    plt.tight_layout()
    plt.show()


gal_b = slider(w.FloatSlider, value=0.04, min=0.025, max=0.1, step=0.005, readout_format=".3f", description="b")
gal_pal = w.Dropdown(options=list(PALETTES), value="leaf", description="palette",
                     style={"description_width": "10em"}, layout=w.Layout(width="28em"))
gal_closed = w.Checkbox(value=True, description="closed venation", layout=w.Layout(width="28em"))
gal_btn = w.Button(description="Draw gallery (~20 s)", button_style="primary")
gal_out = w.Output()


def on_gal(_):
    gal_btn.disabled = True
    with gal_out:
        gal_out.clear_output(wait=True)
        gallery(gal_b.value, gal_pal.value, gal_closed.value)
    gal_btn.disabled = False


gal_btn.on_click(on_gal)
display(w.VBox([gal_b, gal_pal, gal_closed, gal_btn, gal_out]))
if FAST:
    on_gal(None)


## Sources

[1] A. Runions, M. Fuhrer, B. Lane, P. Federl, A.-G. Rolland-Lagan, P. Prusinkiewicz, *Modeling and visualization of leaf venation patterns*, ACM Transactions on Graphics 24(3), 702-711 (SIGGRAPH 2005). Open and closed venation, relative neighbours (eq. 2), Murray widths. <http://algorithmicbotany.org/papers/venation.sig2005.pdf>

[2] D. Hu, D. Cai, *Adaptation and optimization of biological transport networks*, Physical Review Letters 111, 138701 (2013). The adaptive flow network of Recipe B. I read descriptions of this paper in later work, not the paper itself.

[3] E. Katifori, G. J. Szöllősi, M. O. Magnasco, *Damage and fluctuations induce loops in optimal transport networks*, Physical Review Letters 104, 048704 (2010). <https://arxiv.org/abs/0906.0006>. Cost $\sum C^\gamma$: trees for $\gamma < 1$, a uniform sheet above; fluctuating sinks and random damage both give loops.

[4] F. Corson, *Fluctuations and redundancy in optimal transport networks*, Physical Review Letters 104, 048703 (2010). <https://arxiv.org/abs/0905.4947>. Optimal trees assume a stationary flow; fluctuations give hierarchical networks with loops.

[5] C. D. Murray, *The physiological principle of minimum work*, Proceedings of the National Academy of Sciences 12, 207-214 (1926). Radii of a branching vessel obey $r_\text{parent}^n = \sum r_\text{child}^n$, with $n = 3$ in the original.

[6] K. Klemm, E. A. Martens, *Non-local transitions and ground state switching in the self-organization of vascular networks*, arXiv:2407.04120 (first version titled *Optimization dynamics and fluctuations in the self-organization of vascular networks*). The energy $E$, the family of update rules, the moving-sink load with amplitude $\alpha$, and the three-node transition at $\alpha = 1/\sqrt3$. <https://arxiv.org/abs/2407.04120>

What I read: [1] in full text (sections 3 to 6 and its bibliography). [3], [4] and [6] through summaries of their arXiv pages (for [3] and [6], their model sections), not line by line. [2] and [5] I did not read; they are cited from [1]'s bibliography and from later papers that describe them.